# EDA: Telegram-экспорты и подготовка сообщений

Цель: понять состав данных и подготовить три CSV — по одному на каждый экспорт. В результирующие файлы входят датированные сообщения начиная **с 1 июля 2026 года** (включительно). Более ранние записи исключаются. В каждой строке сохраняются чат, дата, автор, текст, ссылка на ответ и локальные пути к вложениям, если они присутствуют. Пути к вложениям указаны относительно каталога соответствующего экспорта.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse, unquote
import json
import re

import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "processed"
CUTOFF = pd.Timestamp("2026-07-01", tz="Europe/Moscow")
EXPORTS = sorted(p for p in DATA_DIR.glob("ChatExport*") if p.is_dir())
print(f"Экспортов: {len(EXPORTS)}")
for path in EXPORTS:
    html_files = sorted(path.glob("messages*.html"))
    print(
        f"- {path.name}: {len(html_files)} HTML-файлов, {sum(p.stat().st_size for p in html_files):,} байт HTML"
    )

## Что находится в экспортах

Экспорт представлен HTML-файлами Telegram. Заголовок страницы задаёт название чата; блоки `message` содержат обычные сообщения и служебные события. Дата берётся из атрибута `title` временной метки, а текст — из отдельного текстового блока, чтобы в него не попадали имя автора, время и подпись к ответу. Локальные ссылки на вложения преобразуются в пути относительно каталога экспорта; внешние ссылки остаются в поле ссылок.

In [ ]:
def local_path_for(href, html_file, export_root):
    if not href:
        return None
    parsed = urlparse(href)
    if parsed.scheme or parsed.netloc or href.startswith("#"):
        return None
    raw = unquote(parsed.path)
    if not raw:
        return None
    resolved = (html_file.parent / raw).resolve()
    try:
        return resolved.relative_to(export_root.resolve()).as_posix()
    except ValueError:
        return None


def parse_message(element, html_file, export_root, chat_name):
    date_node = element.select_one(".pull_right.date[title]")
    date = (
        pd.to_datetime(
            date_node.get("title"), format="%d.%m.%Y %H:%M:%S UTC%z", errors="coerce"
        )
        if date_node
        else pd.NaT
    )
    body = element.select_one(".body")
    text_node = body.select_one(".text") if body else None
    text = text_node.get_text(" ", strip=True) if text_node else ""
    text = re.sub(r"\s+", " ", text).strip()
    sender_node = body.select_one(".from_name") if body else None
    sender = sender_node.get_text(" ", strip=True) if sender_node else ""
    reply_node = body.select_one(".reply_to a[href]") if body else None
    reply_to = ""
    if reply_node:
        match = re.search(
            r"(?:go_to_message|#go_to_message)(\d+)", reply_node.get("href", "")
        )
        reply_to = match.group(1) if match else reply_node.get_text(" ", strip=True)
    file_paths, links = [], []
    if body:
        # Файлами считаем только локальные ссылки из блока вложения;
        # ссылки reply_to и переходы между HTML-страницами не являются медиа.
        for a in body.select(".media_wrap a[href]"):
            href = a.get("href", "").strip()
            local_path = local_path_for(href, html_file, export_root)
            if local_path and local_path not in file_paths:
                file_paths.append(local_path)
            elif (
                urlparse(href).scheme in {"http", "https", "mailto"}
                and href not in links
            ):
                links.append(href)
        # У некоторых медиа нет кликабельной ссылки, поэтому используем src,
        # но только как fallback и только внутри контейнера вложения.
        if not file_paths:
            for img in body.select(".media_wrap img[src]"):
                local_path = local_path_for(img.get("src"), html_file, export_root)
                if local_path and local_path not in file_paths:
                    file_paths.append(local_path)
        for a in body.select(".text a[href]"):
            href = a.get("href", "").strip()
            if (
                urlparse(href).scheme in {"http", "https", "mailto"}
                and href not in links
            ):
                links.append(href)
    classes = element.get("class", [])
    is_service = "service" in classes
    has_media = bool(
        body
        and body.select_one(".media_wrap, .photo, .video, .audio, .document, .sticker")
    )
    if is_service:
        content_type = "service"
    elif text and file_paths:
        content_type = "text_and_file"
    elif file_paths:
        content_type = "file_or_media"
    elif text and has_media:
        content_type = "text_and_media"
    elif has_media:
        content_type = "media"
    elif text:
        content_type = "text"
    elif links:
        content_type = "link"
    else:
        content_type = "other"
    return {
        "chat_name": chat_name,
        "message_id": (element.get("id") or "").replace("message", ""),
        "date": date,
        "sender": sender,
        "text": text,
        "reply_to_message_id": reply_to,
        "file_paths": json.dumps(file_paths, ensure_ascii=False),
        "links": json.dumps(links, ensure_ascii=False),
        "content_type": content_type,
        "is_service": is_service,
        "source_html": html_file.name,
    }


def load_export(export_root):
    records = []
    for html_file in sorted(export_root.glob("messages*.html")):
        soup = BeautifulSoup(
            html_file.read_text(encoding="utf-8", errors="replace"), "html.parser"
        )
        title = soup.select_one(".page_header .text")
        chat_name = title.get_text(" ", strip=True) if title else html_file.stem
        for element in soup.select(".history > .message"):
            records.append(parse_message(element, html_file, export_root, chat_name))
    frame = pd.DataFrame(records)
    if not frame.empty:
        frame["date"] = pd.to_datetime(frame["date"], utc=True).dt.tz_convert(
            "Europe/Moscow"
        )
        frame = frame.sort_values(
            ["chat_name", "date", "message_id"], kind="stable", na_position="last"
        ).reset_index(drop=True)
    return frame


all_exports = {path.name: load_export(path) for path in EXPORTS}
all_messages = pd.concat(
    [df.assign(source_export=name) for name, df in all_exports.items()],
    ignore_index=True,
)
print(f"Всего разобрано строк: {len(all_messages):,}")
print(
    f"Датированных строк: {all_messages.date.notna().sum():,}; без даты: {all_messages.date.isna().sum():,}"
)
print(f"Диапазон дат: {all_messages.date.min()} — {all_messages.date.max()}")
print(f"Уникальных чатов: {all_messages.chat_name.nunique()}")

## Сводка состава данных

In [ ]:
summary = (
    all_messages.groupby("source_export", dropna=False)
    .agg(
        rows=("message_id", "size"),
        chats=("chat_name", "nunique"),
        dated=("date", "count"),
        authors=("sender", lambda s: s[s.ne("")].nunique()),
        with_text=("text", lambda s: s.str.len().gt(0).sum()),
        service_events=("is_service", "sum"),
        with_files=("file_paths", lambda s: s.ne("[]").sum()),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_index()
)
summary

In [ ]:
print("Распределение типа контента:")
display(
    all_messages["content_type"]
    .value_counts(dropna=False)
    .rename_axis("content_type")
    .to_frame("rows")
)
print("Топ авторов:")
display(
    all_messages.loc[all_messages.sender.ne(""), "sender"]
    .value_counts()
    .head(15)
    .rename_axis("sender")
    .to_frame("messages")
)
print("Чаты и источники:")
display(
    all_messages.groupby(["source_export", "chat_name"])
    .size()
    .rename("rows")
    .to_frame()
)

In [ ]:
dated = all_messages.dropna(subset=["date"]).copy()
monthly = (
    dated.assign(month=dated.date.dt.strftime("%Y-%m"))
    .groupby(["source_export", "month"])
    .size()
    .rename("messages")
    .reset_index()
)
display(
    monthly.pivot(index="month", columns="source_export", values="messages")
    .fillna(0)
    .astype(int)
)

In [ ]:
pd.read_csv(
    PROJECT_ROOT / "data/processed/chatexport_2026-10-05_messages_from_2026-07.csv"
)

## Данные для объединения в плоский текст

Для последующего объединения сообщений в линейный текст поля `date`, `sender`, `text`, `file_paths` и `reply_to_message_id` сохранены отдельно. Вложения перечислены JSON-массивом строк: `[]`, если файловых ссылок не найдено, и `[...]`, если они есть. Служебные события также сохранены отдельными строками; это позволяет при необходимости исключить их при формировании плоского корпуса.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_stats = []
for export_name, frame in all_exports.items():
    eligible = frame.loc[frame.date.ge(CUTOFF)].copy()
    eligible = eligible.sort_values(
        ["chat_name", "date", "message_id"], kind="stable"
    ).reset_index(drop=True)
    out_name = (
        re.sub(r"[^A-Za-z0-9_-]+", "_", export_name).strip("_").lower()
        + "_messages_from_2026-07.csv"
    )
    out_path = OUTPUT_DIR / out_name
    eligible.to_csv(out_path, index=False, encoding="utf-8-sig")
    export_stats.append(
        {
            "source_export": export_name,
            "csv": str(out_path),
            "rows_after_cutoff": len(eligible),
            "chats": eligible.chat_name.nunique(),
        }
    )
pd.DataFrame(export_stats)

## Итог

Созданы три CSV-файла — по одному для каждого исходного экспорта. Они содержат только датированные записи от 1 июля 2026 года включительно. В итоговых данных порядок строк внутри каждого чата хронологический; локальные пути к найденным вложениям сохранены в `file_paths`.